# Classificação de Qualidade de Vinhos

In [14]:
# configuração para não exibir os warnings
import warnings
warnings.filterwarnings("ignore")

# Imports necessários
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC


## Carga do Dataset

In [15]:
url = "https://raw.githubusercontent.com/cclguedes/Classificador_Vinhos_Backend/refs/heads/main/dataset/WineQT.csv"

dataset = pd.read_csv(url, delimiter=',')
dataset.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


## Análise Inicial
Inicialmente, foi realizada uma análise exploratória do dataset com o objetivo de compreender sua estrutura, qualidade e características principais.

Por meio da função `info()`, verificou-se que o conjunto de dados possui 1143 registros distribuídos em 13 colunas, todas compostas por variáveis numéricas. Além disso, observou-se que não há valores nulos em nenhuma das variáveis, indicando que o dataset está completo e não requer tratamento de dados ausentes.

Em seguida, foi utilizada a função `describe()` para obter estatísticas descritivas das variáveis, como média, desvio padrão, valores mínimos e máximos. Essa análise permitiu entender a distribuição dos dados e identificar que os valores estão dentro de faixas esperadas, sem indícios de inconsistências ou outliers evidentes.

Por fim, a verificação de valores nulos foi reforçada com a função `isnull().sum()`, confirmando que todas as colunas possuem valores completos. Esse resultado simplifica o processo de pré-processamento, uma vez que não será necessário aplicar técnicas de imputação ou remoção de dados.

Dessa forma, conclui-se que o dataset apresenta boa qualidade e está adequado para as próximas etapas de modelagem.


In [16]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


In [17]:
dataset.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
count,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000,1143.000000
mean,8.311111,0.531339,0.268364,2.532152,0.086933,15.615486,45.914698,0.996730,3.311015,0.657708,10.442111,5.657043,804.969379
std,1.747595,0.179633,0.196686,1.355917,0.047267,10.250486,32.782130,0.001925,0.156664,0.170399,1.082196,0.805824,463.997116
min,4.600000,0.120000,0.000000,0.900000,0.012000,1.000000,6.000000,0.990070,2.740000,0.330000,8.400000,3.000000,0.000000
25%,7.100000,0.392500,0.090000,1.900000,0.070000,7.000000,21.000000,0.995570,3.205000,0.550000,9.500000,5.000000,411.000000
50%,7.900000,0.520000,0.250000,2.200000,0.079000,13.000000,37.000000,0.996680,3.310000,0.620000,10.200000,6.000000,794.000000
75%,9.100000,0.640000,0.420000,2.600000,0.090000,21.000000,61.000000,0.997845,3.400000,0.730000,11.100000,6.000000,1209.500000
max,15.900000,1.580000,1.000000,15.500000,0.611000,68.000000,289.000000,1.003690,4.010000,2.000000,14.900000,8.000000,1597.000000


In [18]:
dataset.isnull().sum()

,0
fixed acidity,0
volatile acidity,0
citric acid,0
residual sugar,0
chlorides,0
free sulfur dioxide,0
total sulfur dioxide,0
density,0
pH,0
sulphates,0


## Transformação do Problema
A variável alvo original do dataset, denominada `quality`, é representada por valores numéricos inteiros que indicam a qualidade do vinho. No entanto, como o objetivo deste trabalho é aplicar técnicas de classificação supervisionada, foi necessário transformar essa variável em categorias discretas.

Para isso, foi definida uma função de categorização que agrupa os valores de qualidade em três classes distintas: "ruim", para valores menores ou iguais a 5; "medio", para valores entre 6 e 7; e "bom", para valores maiores ou iguais a 8. Essa abordagem permite converter o problema originalmente numérico em um problema de classificação binária, uma vez que a variável alvo passa a possuir duas categorias distintas ("ruim" e "bom").

A função foi aplicada a toda a coluna `quality`, gerando uma nova variável chamada `categoria`, que passa a representar a variável alvo do modelo.

Em seguida, foi realizada a análise da distribuição das classes por meio da função `value_counts()`. Observou-se que o dataset apresenta um desbalanceamento entre as classes, com predominância das categorias "medio" e "ruim", e uma quantidade significativamente menor de exemplos da classe "bom". Essa característica pode impactar o desempenho dos modelos, especialmente na capacidade de generalização para a classe minoritária.

Por fim, a coluna original `quality` foi removida do dataset, garantindo que o modelo não utilize diretamente a variável numérica original durante o treinamento. Essa etapa é fundamental para evitar vazamento de informação (data leakage) e assegurar que o modelo aprenda exclusivamente a partir das variáveis independentes.

Dessa forma, o problema foi adequadamente transformado para um cenário de classificação, permitindo a aplicação dos algoritmos propostos.


In [19]:
def categorizar(q):
    if q <= 5:
        return "ruim"
    else:
        return "bom"

dataset["categoria"] = dataset["quality"].apply(categorizar)

In [20]:
dataset = dataset.drop(columns=["quality"])

## Separação das Variáveis

Nesta etapa, o dataset foi dividido em variáveis independentes (X) e variável alvo (y), conforme exigido em problemas de aprendizado supervisionado.

As variáveis independentes (X) correspondem às características físico-químicas do vinho, que serão utilizadas como entrada para o modelo. Para isso, foram removidas as colunas `categoria` e `Id`, sendo a primeira a variável alvo e a segunda apenas um identificador, sem relevância para o processo de aprendizado.

A variável alvo (y) foi definida como a coluna `categoria`, que representa a classificação da qualidade do vinho em duas classes distintas: "ruim" e "bom".

Essa separação é fundamental, pois permite que o modelo aprenda a relação entre as características dos dados (X) e a variável que se deseja prever (y).

In [21]:
X = dataset.drop(columns=["categoria", "Id"])
y = dataset["categoria"]

## Separação em Conjunto de Treino e Teste

Nesta etapa, os dados foram divididos em dois subconjuntos: treino e teste, utilizando a técnica de holdout.

Foi definida uma proporção de 80% dos dados para treinamento e 20% para teste. O conjunto de treino é utilizado para ajustar o modelo, enquanto o conjunto de teste é reservado para avaliar seu desempenho em dados não vistos, permitindo uma estimativa mais realista da capacidade de generalização.

A divisão foi realizada de forma estratificada, por meio do parâmetro `stratify=y`, garantindo que a proporção das classes seja mantida em ambos os conjuntos. Essa abordagem é especialmente importante devido ao desbalanceamento observado nas classes do dataset.

Além disso, foi definida uma semente aleatória (`random_state=7`) para garantir a reprodutibilidade dos resultados.

Por fim, foi configurada a validação cruzada estratificada (Stratified K-Fold) com 10 partições, que será utilizada posteriormente para avaliar os modelos de forma mais robusta.

In [22]:
test_size = 0.20
seed = 7

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=test_size,
    random_state=seed,
    stratify=y
)

kfold = StratifiedKFold(n_splits=10, shuffle=True, random_state=seed)

## Modelagem
Nesta etapa, foram aplicados diferentes algoritmos de classificação supervisionada com o objetivo de identificar qual modelo apresenta melhor desempenho para o problema proposto.

Foram selecionados quatro algoritmos clássicos amplamente utilizados em tarefas de classificação: K-Nearest Neighbors (KNN), Árvore de Decisão (CART), Naive Bayes (NB) e Support Vector Machine (SVM). A escolha desses modelos permite comparar abordagens distintas, baseadas em diferentes princípios, como distância, regras de decisão, probabilidade e separação por margens.

Para avaliar o desempenho dos modelos de forma mais robusta, foi utilizada a técnica de validação cruzada estratificada (Stratified K-Fold), com 10 partições. Essa abordagem garante que a proporção das classes seja mantida em cada divisão dos dados, o que é especialmente importante devido ao desbalanceamento observado no dataset.

A métrica adotada para avaliação foi a acurácia, que mede a proporção de previsões corretas realizadas pelo modelo. Cada algoritmo foi treinado e avaliado utilizando a função `cross_val_score`, permitindo obter uma estimativa mais confiável de desempenho médio.

Os resultados obtidos indicaram que o modelo Naive Bayes apresentou o melhor desempenho médio entre os algoritmos testados, seguido pelos demais modelos com resultados ligeiramente inferiores. Essa comparação permite identificar o modelo mais adequado para o problema, considerando as características do dataset.

Dessa forma, a etapa de modelagem possibilitou não apenas o treinamento dos modelos, mas também uma análise comparativa de desempenho, contribuindo para a escolha de uma abordagem mais eficaz.


In [23]:
models = []
models.append(("KNN", KNeighborsClassifier()))
models.append(("CART", DecisionTreeClassifier()))
models.append(("NB", GaussianNB()))
models.append(("SVM", SVC()))

for name, model in models:
    results = cross_val_score(model, X_train, y_train, cv=kfold, scoring='accuracy')
    print(f"{name}: {results.mean():.4f}")

KNN: 0.6619
CART: 0.7231
NB: 0.7221
SVM: 0.6313


## Avaliação Final

Após a etapa de modelagem e comparação entre os algoritmos, foi selecionado um modelo para avaliação final utilizando o conjunto de teste, que não foi utilizado durante o treinamento.

Durante a etapa de validação cruzada, observou-se que o modelo Naive Bayes apresentou o melhor desempenho médio entre os algoritmos avaliados.

Antes do treinamento, os dados foram padronizados com o uso do StandardScaler, garantindo que todas as variáveis estivessem na mesma escala. Essa etapa é especialmente importante para algoritmos sensíveis à magnitude dos dados, como o SVM.

O modelo selecionado foi então treinado com os dados de treino e posteriormente utilizado para realizar previsões no conjunto de teste. A avaliação foi feita por meio da métrica de acurácia, permitindo medir a capacidade do modelo de generalizar para dados não vistos.

O resultado obtido apresentou uma acurácia de aproximadamente 0.71, indicando que o modelo é capaz de classificar corretamente cerca de 71% das amostras. Esse desempenho é considerado satisfatório, levando em conta a complexidade do problema e o desbalanceamento das classes.

Dessa forma, a avaliação final demonstra que o modelo possui um bom desempenho geral e pode ser utilizado para realizar previsões em novos dados.

In [24]:
# padronização dos dados
scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)

# modelo final (SVM)
model = SVC()
model.fit(X_train_scaled, y_train)

# avaliação no conjunto de teste
X_test_scaled = scaler.transform(X_test)
predictions = model.predict(X_test_scaled)

print("Accuracy:", accuracy_score(y_test, predictions))

Accuracy: 0.7248908296943232


## Simulação
Após o treinamento e avaliação do modelo, foi realizada uma simulação utilizando um novo conjunto de dados, com o objetivo de demonstrar sua aplicação prática.

Para isso, foi criado um exemplo fictício contendo valores das variáveis físico-químicas de um vinho. Esses dados foram organizados no mesmo formato utilizado durante o treinamento do modelo.

Antes de realizar a predição, os dados de entrada foram submetidos ao mesmo processo de padronização aplicado ao conjunto de treino, utilizando o StandardScaler previamente ajustado. Essa etapa é fundamental para garantir consistência entre os dados utilizados no treinamento e aqueles utilizados na inferência.

Em seguida, o modelo treinado foi utilizado para prever a classe correspondente ao novo exemplo. O resultado da predição indica a categoria de qualidade estimada para o vinho, podendo ser "ruim" ou "bom".

Essa etapa demonstra a capacidade do modelo de ser utilizado em cenários reais, permitindo a classificação de novos dados a partir das características fornecidas.


In [25]:
import numpy as np

novo_vinho = np.array([[7.4, 0.7, 0.0, 1.9, 0.076, 11.0, 34.0, 0.9978, 3.51, 0.56, 9.4]])

novo_vinho = scaler.transform(novo_vinho)
pred = model.predict(novo_vinho)

print("Classe prevista:", pred[0])

Classe prevista: ruim


## Conclusão

Neste trabalho, foi possível aplicar técnicas clássicas de machine learning para classificação da qualidade de vinhos, passando por todas as etapas do processo, desde a análise dos dados até a avaliação do modelo.

Os resultados obtidos demonstram que os algoritmos utilizados são capazes de capturar padrões relevantes nos dados, permitindo a construção de um modelo com bom desempenho.

Além disso, a comparação entre diferentes abordagens possibilitou uma análise mais completa, evidenciando as características e limitações de cada modelo.

In [26]:
import joblib

joblib.dump(model, "model.pkl")
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']